# Una función que el modelo puede pedir

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un modelo de lenguaje solo produce texto: no sabe la hora, no ve una planilla y no calcula con exactitud. Una herramienta es una función que el programa ejecuta cuando el modelo lo pide, y el modelo decide si pedirla y con qué argumentos leyendo su descripción. |
| **Producto** | El ciclo de una herramienta armado a mano, una medición de cómo la descripción cambia los argumentos y una herramienta propia con su descripción comprobada. |
| **Recursos** | Ollama con `granite4:latest`. La ejecución completa lleva menos de dos minutos. Sin claves. |

Una cooperativa agropecuaria registra la lluvia de cada parcela en milímetros. Quien arma el informe necesita litros, que dependen de la superficie. Un asistente de lenguaje podría hacer la cuenta, pero los modelos fallan con la aritmética y, sobre todo, con las unidades.

**¿Cómo se conecta un modelo con una función de Python, y por qué la descripción de la función importa tanto como su código?**

> **Cómo se trabaja.** En clase se presenta cada tema con el código ya escrito. Después lo explorás por tu cuenta: ejecutás las celdas, leés las notas **Para mirar** y cambiás valores para ver qué pasa. Al final de cada bloque hay un **microlaboratorio**: una tabla te dice qué líneas completar y con qué, y la celda siguiente comprueba si quedó bien.

## Los datos

| Campo | Dato |
|---|---|
| Fuente | Ninguna: el cuaderno usa un solo caso numérico **inventado** (12 mm de lluvia sobre una parcela de 2500 m²) |
| Equivalencia | 1 mm de lluvia equivale a 1 litro por metro cuadrado, es decir, 10 000 litros por hectárea. La respuesta correcta del caso es **30 000 litros** (12 mm sobre 0,25 hectáreas). |
| Limitaciones | Un solo caso, un solo modelo y una sola unidad en juego. Muestra el mecanismo y un tipo de error frecuente; no estima qué tan seguido falla un modelo. |

In [1]:
import utils  # va primero: ordena los avisos de las librerias que siguen
import ollama

utils.comprobar_modelo_local()
utils.versiones(["ollama", "smolagents"])

Ollama responde en http://localhost:11434 y tiene granite4:latest.


{'python': '3.12.13', 'ollama': '0.6.3', 'smolagents': '1.26.0'}

## 1. Sin herramienta

Primero, la pregunta tal cual. Pedimos un número y su unidad, y limitamos la respuesta a 250 tokens.

In [2]:
pregunta = (
    "Cayeron 12 mm de lluvia sobre una parcela de 2500 metros cuadrados. "
    "¿Cuántos litros fueron? Respondé con un número y la unidad."
)

sin_herramienta = ollama.chat(
    model=utils.MODELO_LOCAL,
    messages=[{"role": "user", "content": pregunta}],
    options={"temperature": 0, "num_predict": 250},
)
print(sin_herramienta.message.content)

Para calcular el volumen de agua que cayó, primero debemos convertir los milímetros de lluvia a metros, ya que la superficie está dada en metros cuadrados.

1 mm = 0.001 metros

Por lo tanto, 12 mm = 12 * 0.001 = 0.012 metros

Ahora, aplicamos el cálculo del volumen:

Volumen (en metros cúbicos) = Altura de la lluvia (en metros) * Área (en metros cuadrados)

Volumen = 0.012 metros * 2500 metros²

Volumen = 30 metros³

Finalmente, convirtiendo metros cúbicos a litros:

1 metro cúbico = 1000 litros

Por lo tanto, 30 metros cúbicos = 30 * 1000 = 30000 litros

La respuesta es: **30.000** litros.


> **Para mirar.**
>
> - Esta vez el modelo resolvió bien el caso (30 000 litros), a costa de un razonamiento largo: pasó de milímetros a metros y de metros cúbicos a litros.
> - Que acierte una vez no garantiza que acierte siempre ni con otra unidad. Cuando la cuenta la hace una función, es exacta; cuando la hace el modelo, es una respuesta más que hay que verificar.

## 2. El modelo propone, el programa ejecuta

Escribimos la cuenta como una función de Python **con una descripción** (docstring): qué hace, qué es cada argumento y en qué unidad. Se la pasamos al modelo con el parámetro `tools`. El modelo no ejecuta nada: **devuelve un pedido** con el nombre de la función y los argumentos que elige.

In [3]:
def convertir_mm_a_litros(milimetros: float, hectareas: float) -> float:
    """Convierte una lluvia en milimetros a litros sobre una superficie.

    Args:
        milimetros: Lluvia caida, en milimetros.
        hectareas: Superficie de la parcela, en hectareas (1 hectarea son
            10000 metros cuadrados).

    Returns:
        Litros de agua caidos sobre toda la superficie.
    """
    return milimetros * hectareas * 10000


primera = ollama.chat(
    model=utils.MODELO_LOCAL,
    messages=[{"role": "user", "content": pregunta}],
    tools=[convertir_mm_a_litros],
    options={"temperature": 0},
)

print("Texto del modelo:", repr(primera.message.content))
for pedido in primera.message.tool_calls:
    print("Función pedida:", pedido.function.name)
    print("Argumentos:", dict(pedido.function.arguments))

Texto del modelo: ''
Función pedida: convertir_mm_a_litros
Argumentos: {'hectareas': 0.25, 'milimetros': 12}


> **Para mirar.**
>
> - `content` está vacío: el modelo no respondió con texto. Devolvió un pedido con el nombre de la función y los argumentos `milimetros=12` y `hectareas=0.25`.
> - Convirtió por su cuenta 2500 m² en 0,25 hectáreas: lo hizo leyendo la descripción, no el código.
> - Todavía no se ejecutó nada: lo ejecuta nuestro programa, cuando quiere.

## 3. Cerrar el ciclo

Para que el modelo use el resultado, hay que **devolvérselo**: ejecutamos la función con los argumentos que pidió y agregamos a la conversación un mensaje con el rol `tool`. Una segunda llamada produce la respuesta final. Eso es, en pequeño, lo que hace un agente.

In [4]:
historial = [{"role": "user", "content": pregunta}]
historial.append(primera.message)

for pedido in primera.message.tool_calls:
    argumentos = dict(pedido.function.arguments)
    resultado = convertir_mm_a_litros(**argumentos)
    historial.append({
        "role": "tool",
        "content": str(resultado),
        "tool_name": pedido.function.name,
    })

segunda = ollama.chat(
    model=utils.MODELO_LOCAL,
    messages=historial,
    tools=[convertir_mm_a_litros],
    options={"temperature": 0},
)

for mensaje in historial:
    rol = mensaje["role"] if isinstance(mensaje, dict) else mensaje.role
    print("Mensaje con rol:", rol)
print()
print("Respuesta final:", segunda.message.content)

Mensaje con rol: user
Mensaje con rol: assistant
Mensaje con rol: tool

Respuesta final: La lluvia cayó en **30 000 L** sobre la parcela de 2 500 m².


> **Para mirar.**
>
> - La conversación tiene tres mensajes: `user` (la pregunta), `assistant` (el pedido de la herramienta) y `tool` (el resultado que devolvimos). Con el rol `tool` el modelo se entera del resultado.
> - La respuesta final cita los 30 000 litros que calculó la función.
> - Quien decide qué se ejecuta es el programa, no el modelo: en ese punto se podría validar, limitar o rechazar el pedido.

### Microlaboratorio 1 — El ciclo en una función

**Tarea.** Escribir la función que hace una ronda completa: pedir, ejecutar y responder.  
**Producto.** `resolver`, comprobada con la pregunta de la lluvia.  
**Antes de empezar.** Sección 3.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `resultado` | lo que devuelve la función pedida, ejecutada con los argumentos del pedido | `disponibles[nombre](**argumentos)` |
| (2) `"tool_name"` | el nombre de la función que produjo el resultado | la variable `nombre` |

**Comprobación.** La respuesta final tiene que mencionar 30 000 litros, y el último mensaje del historial tiene que decir qué herramienta lo produjo.

> **Ayuda 1.** En la sección 3, `convertir_mm_a_litros(**argumentos)` hace la llamada; acá la función se busca por nombre en el diccionario `disponibles`.  
> **Ayuda 2.** `**argumentos` convierte un diccionario en argumentos con nombre.

In [5]:
def resolver(pregunta: str, herramientas: list) -> dict:
    """Responde una pregunta dejando que el modelo pida herramientas.

    Args:
        pregunta: Pregunta de la persona.
        herramientas: Funciones de Python con descripcion (docstring).

    Returns:
        Diccionario con "respuesta" y "historial".
    """
    disponibles = {}
    for funcion in herramientas:
        disponibles[funcion.__name__] = funcion
    historial = [{"role": "user", "content": pregunta}]
    pedido_del_modelo = ollama.chat(
        model=utils.MODELO_LOCAL,
        messages=historial,
        tools=herramientas,
        options={"temperature": 0},
    )
    historial.append(pedido_del_modelo.message)
    for pedido in pedido_del_modelo.message.tool_calls or []:
        nombre = pedido.function.name
        argumentos = dict(pedido.function.arguments)
        resultado = ""  # COMPLETAR (1)
        historial.append({
            "role": "tool",
            "content": str(resultado),
            "tool_name": "",  # COMPLETAR (2)
        })
    final = ollama.chat(
        model=utils.MODELO_LOCAL,
        messages=historial,
        tools=herramientas,
        options={"temperature": 0},
    )
    return {"respuesta": final.message.content, "historial": historial}


con_resolver = resolver(pregunta, [convertir_mm_a_litros])
texto_final = con_resolver["respuesta"].replace(" ", "").replace(",", "")
utils.comprobar(
    "30000" in texto_final.replace(".", "").replace("\u202f", ""),
    "revisa (1): la respuesta final tiene que mencionar 30000 litros",
)
ultimo = con_resolver["historial"][-1]
utils.comprobar(
    ultimo["tool_name"] == "convertir_mm_a_litros",
    "revisa (2): el mensaje tool tiene que llevar el nombre de la funcion",
)

Todavia no: revisa (1): la respuesta final tiene que mencionar 30000 litros
Todavia no: revisa (2): el mensaje tool tiene que llevar el nombre de la funcion


A partir de acá usamos `utils.resolver_con_herramientas`, que hace lo mismo que tu función y además devuelve la lista de pedidos del modelo.

## 4. La descripción es el contrato

El modelo nunca ve el código de la función: ve **el nombre, los argumentos y la descripción**. Probamos tres versiones de la misma función, con el mismo código y distinta descripción, y repetimos el pedido con cinco semillas (la **semilla** fija el azar de cada repetición, así que vas a obtener lo mismo que se muestra acá):

- **detallada**: dice que la superficie va en hectáreas y cuánto vale una hectárea;
- **escueta**: dice solo «Superficie.»;
- **vaga**: «Hace cosas.», con argumentos llamados `a` y `b`.

In [6]:
def con_descripcion_escueta(milimetros: float, superficie: float) -> float:
    """Convierte lluvia a litros.

    Args:
        milimetros: Lluvia.
        superficie: Superficie.

    Returns:
        Litros.
    """
    return milimetros * superficie * 10000


def vaga(a: float, b: float) -> float:
    """Hace cosas."""
    return a * b * 10000


versiones_de_la_funcion = {
    "detallada": convertir_mm_a_litros,
    "escueta": con_descripcion_escueta,
    "vaga": vaga,
}

LITROS_CORRECTOS = 30000
filas_de_descripcion = []
for nombre_de_version, funcion in versiones_de_la_funcion.items():
    correctas = 0
    for semilla in (1, 2, 3, 4, 5):
        respuesta = ollama.chat(
            model=utils.MODELO_LOCAL,
            messages=[{"role": "user", "content": pregunta}],
            tools=[funcion],
            options={"temperature": 0.7, "seed": semilla},
        )
        pedidos = respuesta.message.tool_calls or []
        if len(pedidos) == 0:
            litros = None
        else:
            litros = funcion(**dict(pedidos[0].function.arguments))
        if litros == LITROS_CORRECTOS:
            correctas += 1
        argumentos_pedidos = None
        if pedidos:
            argumentos_pedidos = dict(pedidos[0].function.arguments)
        filas_de_descripcion.append({
            "descripción": nombre_de_version,
            "semilla": semilla,
            "argumentos": argumentos_pedidos,
            "litros": litros,
        })
    print(f"{nombre_de_version}: {correctas} de 5 con el resultado correcto")

utils.tabla(filas_de_descripcion)

detallada: 4 de 5 con el resultado correcto


escueta: 0 de 5 con el resultado correcto


vaga: 0 de 5 con el resultado correcto


,descripción,semilla,argumentos,litros
0,detallada,1,"{'hectareas': 0.25, 'milimetros': 12}",3.000000e+04
1,detallada,2,"{'hectareas': 0.25, 'milimetros': 12}",3.000000e+04
2,detallada,3,"{'hectareas': 0.25, 'milimetros': 12}",3.000000e+04
3,detallada,4,"{'hectareas': 25, 'milimetros': 12}",3.000000e+06
4,detallada,5,"{'hectareas': 0.25, 'milimetros': 12}",3.000000e+04
5,escueta,1,"{'milimetros': 12, 'superficie': 2500}",3.000000e+08
6,escueta,2,"{'milimetros': 12, 'superficie': 2500}",3.000000e+08
7,escueta,3,"{'milimetros': 12, 'superficie': 2500}",3.000000e+08
8,escueta,4,"{'milimetros': 12, 'superficie': 2500}",3.000000e+08
9,escueta,5,"{'milimetros': 12, 'superficie': 2500}",3.000000e+08


> **Para mirar.**
>
> - **detallada**: 4 de 5 correctas. En la semilla 4 pidió 25 hectáreas en lugar de 0,25 y el resultado fue 3 000 000 de litros, cien veces más.
> - **escueta**: 0 de 5. Las cinco veces pasó 2500 como `superficie` y el resultado fue 300 000 000 de litros, diez mil veces más.
> - **vaga**: 0 de 5. En dos semillas no pidió la función y en las otras tres inventó valores para `a` y `b` con resultados de un millón y diez millones de veces el correcto.
> - El código era idéntico en las tres versiones: cambió solo lo que el modelo lee.

## 5. Cuándo no pedir una herramienta

Un modelo con herramientas también decide **no usarlas**. Le damos dos funciones (la hora y la conversión) y tres preguntas: una para cada una y una que no necesita ninguna.

In [7]:
def hora_actual() -> str:
    """Devuelve la fecha y la hora actuales del sistema.

    Returns:
        Fecha y hora en formato texto.
    """
    return "2026-10-23 18:00"


preguntas = [
    "¿Qué hora es?",
    "Cayeron 5 mm sobre 2 hectáreas, ¿cuántos litros fueron?",
    "¿Cuál es la capital de la provincia de Santa Fe?",
]

filas_de_decision = []
for texto in preguntas:
    respuesta = ollama.chat(
        model=utils.MODELO_LOCAL,
        messages=[{"role": "user", "content": texto}],
        tools=[hora_actual, convertir_mm_a_litros],
        options={"temperature": 0},
    )
    pedidos = respuesta.message.tool_calls or []
    if len(pedidos) == 0:
        decision = "ninguna: respondió directamente"
    else:
        decision = pedidos[0].function.name
    filas_de_decision.append({
        "pregunta": texto,
        "herramienta pedida": decision,
    })

utils.tabla(filas_de_decision)

,pregunta,herramienta pedida
0,¿Qué hora es?,hora_actual
1,"Cayeron 5 mm sobre 2 hectáreas, ¿cuántos litros fueron?",convertir_mm_a_litros
2,¿Cuál es la capital de la provincia de Santa Fe?,ninguna: respondió directamente


> **Para mirar.**
>
> - Pidió la función que correspondía a cada pregunta y no pidió ninguna para la capital de Santa Fe, que respondió directamente.
> - Elegir qué herramienta usar, o ninguna, también depende de las descripciones.
> - Con tres preguntas no sabemos qué tan seguido acierta esa elección.

### Microlaboratorio 2 — Escribir la descripción

**Tarea.** Completar la descripción del argumento `hectareas` para que el modelo pase la unidad correcta.  
**Producto.** La función `litros_de_lluvia` con su descripción, comprobada con tres superficies.  
**Antes de empezar.** Sección 4.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) la descripción de `hectareas` dentro del docstring | una frase que diga **en qué unidad** va la superficie y **cuánto vale** esa unidad en metros cuadrados | texto, como en la versión detallada de la sección 4 |

**Comprobación.** Con temperatura `0` y tres preguntas (2500, 4000 y 10 000 m²), el modelo tiene que pedir 0,25, 0,4 y 1 hectáreas.

> **Ayuda 1.** Mirá la versión `detallada` de la sección 4: la unidad está escrita entre paréntesis.  
> **Ayuda 2.** El modelo no sabe cuántos metros cuadrados tiene una hectárea a menos que se lo digas, o que lo recuerde sin garantías.

In [8]:
# COMPLETAR (1): cambiar la descripcion de hectareas en el docstring.
def litros_de_lluvia(milimetros: float, hectareas: float) -> float:
    """Convierte una lluvia en milimetros a litros sobre una superficie.

    Args:
        milimetros: Lluvia caida, en milimetros.
        hectareas: Superficie.

    Returns:
        Litros de agua caidos sobre toda la superficie.
    """
    return milimetros * hectareas * 10000


superficies = [
    (
        "Cayeron 12 mm de lluvia sobre una parcela de 2500 metros "
        "cuadrados. ¿Cuántos litros fueron?",
        0.25,
    ),
    (
        "Cayeron 8 mm de lluvia sobre una parcela de 4000 m2. "
        "¿Cuántos litros fueron?",
        0.4,
    ),
    (
        "Llovieron 20 mm sobre un lote de 10000 metros cuadrados. "
        "¿Cuántos litros cayeron?",
        1.0,
    ),
]

correctas = 0
for texto_de_la_pregunta, hectareas_esperadas in superficies:
    respuesta = ollama.chat(
        model=utils.MODELO_LOCAL,
        messages=[{"role": "user", "content": texto_de_la_pregunta}],
        tools=[litros_de_lluvia],
        options={"temperature": 0},
    )
    pedidos = respuesta.message.tool_calls or []
    if len(pedidos) > 0:
        argumentos = dict(pedidos[0].function.arguments)
        pedida = argumentos.get("hectareas", 0)
        if abs(pedida - hectareas_esperadas) < 0.001:
            correctas += 1

print(f"Superficies bien pedidas: {correctas} de {len(superficies)}")
utils.comprobar(
    correctas == len(superficies),
    "revisa (1): decir en que unidad va hectareas y cuanto vale una",
)

Superficies bien pedidas: 1 de 3
Todavia no: revisa (1): decir en que unidad va hectareas y cuanto vale una


## Qué te llevás

- Una herramienta es una **función con descripción**. El modelo propone qué función llamar y con qué argumentos; el programa la ejecuta y le devuelve el resultado. Ese ciclo, repetido, es un agente.
- El modelo decide **leyendo la descripción**, no el código: la misma función con distintas descripciones produjo resultados correctos, errores por un factor de 10 000 o argumentos al azar.
- Una herramienta bien descripta no garantiza que se use bien: hay que **medir con casos conocidos** y decidir qué se automatiza.

**Sigue:** cómo un agente repite este ciclo varias veces, escribe código para resolver una tarea y qué pasa cuando la herramienta le devuelve un resultado difícil de usar (cuaderno 02).